# Physics-Enhanced Underwater Acoustic Signal Synthesis

## Generating underwater accoustic signals for Submarines and Torpedos - V2

## Utils

In [1]:
import numpy as np

def rng_from_seed(seed: int | None):
    return np.random.default_rng(seed if seed is not None else np.random.SeedSequence().entropy)

def tukey_mask(f, f_lo, f_hi, roll=0.1):
    """
    Smooth band mask in frequency domain using a cosine ramp.
    f: frequency vector (Hz)
    f_lo, f_hi: passband edges
    roll: fraction of transition width
    """
    mask = np.zeros_like(f, dtype=float)
    if f_hi <= f_lo:
        return mask
    width = max((f_hi - f_lo) * roll, 1e-6)
    # rise
    idx1 = (f >= (f_lo - width)) & (f < f_lo)
    mask[idx1] = 0.5 * (1 + np.cos(np.pi * (f_lo - f[idx1]) / width))
    # flat
    idx2 = (f >= f_lo) & (f <= f_hi)
    mask[idx2] = 1.0
    # fall
    idx3 = (f > f_hi) & (f <= (f_hi + width))
    mask[idx3] = 0.5 * (1 + np.cos(np.pi * (f[idx3] - f_hi) / width))
    return mask

def thorp_absorption_dB_per_m(f_hz: np.ndarray) -> np.ndarray:
    """
    Thorp-like absorption approximation (dB/m) vs frequency.
    Roughly valid from a few hundred Hz to tens of kHz.
    """
    f = np.maximum(f_hz, 1e-6) / 1000.0  # kHz
    a_db_per_km = 0.11 * (f**2 / (1 + f**2)) + 44 * (f**2 / (4100 + f**2)) + 2.75e-4 * f**2 + 0.003
    return a_db_per_km / 1000.0  # dB/m

def db_to_lin(db):
    return 10 ** (db / 20.0)

def lin_to_db(lin):
    lin = np.maximum(lin, 1e-20)
    return 20 * np.log10(lin)


## Sensors

In [2]:
import numpy as np
from scipy.signal import butter, sosfiltfilt

def bandlimit(x, fs, lo, hi, order=6):
    """Apply bandpass filter."""
    lo = max(1.0, lo)
    hi = min(0.49*fs, hi)
    sos = butter(order, [lo, hi], btype="band", fs=fs, output="sos")
    return sosfiltfilt(sos, x)

def wenz_ambient_noise(f_hz, sea_state, shipping_level=5):
    """
    Generate ambient noise spectrum based on Wenz curves.
    
    Parameters:
    - f_hz: frequency array in Hz
    - sea_state: 0-6 (Beaufort scale approximation)
    - shipping_level: 0-7 (0=no shipping, 7=heavy shipping)
    
    Returns noise level in dB re 1 µPa²/Hz
    """
    f_khz = np.maximum(f_hz / 1000.0, 0.001)
    
    # Turbulence (very low freq, <10 Hz) - not modeled for typical sonar bands
    
    # Shipping noise (10-300 Hz): ~ 76 - 60*log10(f_kHz) + shipping_factor
    shipping_factor = (7 - shipping_level) * 5  # 0 to 35 dB reduction
    N_ship = 76 - 60*np.log10(f_khz) - shipping_factor
    
    # Wind-dependent noise (>300 Hz): dominant at high frequencies
    # Wenz: NL(f, wind) ≈ 50 + 7.5*wind^0.5 + 20*log10(f) - 40*log10(f+0.4)
    wind_speed_knots = sea_state * 5  # Rough conversion
    N_wind = 50 + 7.5*np.sqrt(wind_speed_knots) + 20*np.log10(f_khz) - 40*np.log10(f_khz + 0.4)
    
    # Thermal noise (high frequencies, >50 kHz) - usually negligible
    N_thermal = -15 + 20*np.log10(f_khz)
    
    # Combine using energy summation (10^(NL/10))
    N_total_linear = 10**(N_ship/10) + 10**(N_wind/10) + 10**(N_thermal/10)
    N_total_dB = 10 * np.log10(N_total_linear)
    
    return N_total_dB

def generate_ambient_noise(n, fs, sea_state, shipping_level, rng):
    """
    Generate realistic ambient noise with Wenz spectrum.
    """
    # Generate white noise
    noise = rng.standard_normal(n).astype(np.float32)
    
    # Apply Wenz spectrum shaping
    N = np.fft.rfft(noise)
    f = np.fft.rfftfreq(n, 1/fs)
    
    # Get Wenz spectrum (dB re 1 µPa²/Hz)
    spectrum_dB = wenz_ambient_noise(f, sea_state, shipping_level)
    
    # Convert to linear magnitude (relative scaling)
    # Normalize to reference frequency (e.g., 100 Hz)
    f_ref_idx = np.argmin(np.abs(f - 100.0))
    spectrum_dB_norm = spectrum_dB - spectrum_dB[f_ref_idx]
    spectrum_linear = 10 ** (spectrum_dB_norm / 20.0)
    
    # Apply shaping
    N_shaped = N * spectrum_linear
    noise_shaped = np.fft.irfft(N_shaped, n)
    
    # Normalize
    noise_shaped /= (np.std(noise_shaped) + 1e-12)
    
    return noise_shaped.astype(np.float32)

def add_ambient_and_sensor_noise(x, fs, class_name, sea_state, snr_db, shipping_level=5, rng=None):
    """
    Add realistic ambient noise (Wenz curves) + white sensor noise to reach target SNR.
    SNR is measured in the signal's primary frequency band.
    
    NOTE: This is for Stage 1 validation only. In Stage 2, Bellhop will handle 
    ambient noise more accurately with proper spatial modeling.
    """
    n = len(x)
    
    # Define signal band for SNR calculation
    if class_name == "submarine":
        band = (20.0, 800.0)
    else:
        band = (300.0, 8000.0)
    
    # Generate ambient noise with Wenz spectrum
    amb = generate_ambient_noise(n, fs, sea_state, shipping_level, rng)
    
    # Bandlimit ambient to relevant frequencies (slight filtering for realism)
    amb = bandlimit(amb, fs, max(1, band[0]*0.5), min(fs*0.49, band[1]*1.5))
    
    # Add white sensor self-noise (thermal, electronic)
    sens = rng.standard_normal(n).astype(np.float32) * 0.01
    
    # Total noise
    noise = amb + sens
    
    # Calculate RMS in signal band for accurate SNR
    sos = butter(6, [band[0], band[1]], btype="band", fs=fs, output="sos")
    x_filt = sosfiltfilt(sos, x)
    noise_filt = sosfiltfilt(sos, noise)
    
    sig_rms = np.sqrt(np.mean(x_filt**2) + 1e-12)
    noise_rms = np.sqrt(np.mean(noise_filt**2) + 1e-12)
    
    # Scale noise to achieve target SNR
    target_noise_rms = sig_rms / (10 ** (snr_db/20.0))
    scale = target_noise_rms / (noise_rms + 1e-12)
    noise *= scale
    
    # Add noise to signal
    y = x + noise
    
    # Normalize to prevent clipping
    y /= (max(1.0, np.max(np.abs(y)) * 1.05))
    
    return y.astype(np.float32)

def sea_state_surface_loss(sea_state):
    """
    Calculate additional surface reflection loss due to sea state.
    Returns loss in dB per surface bounce.
    
    Physics: Rough surface scattering increases with wave height.
    Loss ≈ 0.5 * SS^2 dB per bounce (empirical)
    """
    return 0.5 * sea_state**2


## Sources

In [3]:
from dataclasses import dataclass
import numpy as np


@dataclass
class ClassParams:
    blades_lo: int; blades_hi: int
    rpm_lo: float; rpm_hi: float
    broad: tuple; hub: tuple; tip: tuple
    alpha_broad: tuple; alpha_hub: tuple
    harmonics: int
    tip_rate_base_hz: tuple  # Base cavitation rate at threshold RPM
    tip_rpm_threshold: float  # RPM above which cavitation increases
    tip_rate_exponent: float  # How quickly cavitation increases with RPM
    tip_dur_ms: tuple
    rpm_drift_pct: tuple
    rpm_drift_period_s: tuple
    rpm_walk_std: float  # Random walk standard deviation
    transient_prob: float  # Probability of speed change event

def colored_noise(alpha, fs, band, n, rng):
    """
    Generates band-limited noise with a 1/f^α spectral shape.
    """
    x = rng.standard_normal(n)
    X = np.fft.rfft(x)
    f = np.fft.rfftfreq(n, 1/fs)
    shape = 1.0 / np.maximum(f, 1e-6) ** (alpha/2.0)
    mask = tukey_mask(f, band[0], band[1], roll=0.1)
    Y = X * shape * mask
    y = np.fft.irfft(Y, n=n)
    y /= (np.std(y) + 1e-12)
    return y

def bursty_band_noise(fs, band, n, events, dur_range_s, rng):
    """Creates intermittent, band-limited noise for cavitation bursts."""
    y = np.zeros(n, dtype=float)
    f = np.fft.rfftfreq(n, 1/fs)
    env = np.zeros(n, dtype=float)
    for _ in range(events):
        dur = rng.uniform(*dur_range_s)
        L = int(max(1, dur*fs))
        start = rng.integers(0, max(1, n - L))
        window = 0.5 - 0.5*np.cos(2*np.pi*np.arange(L)/max(L-1,1))
        env[start:start+L] += window
    env = np.clip(env, 0.0, 1.0)
    x = rng.standard_normal(n)
    X = np.fft.rfft(x)
    mask = tukey_mask(f, band[0], band[1], roll=0.08)
    Y = X * mask
    z = np.fft.irfft(Y, n=n)
    z /= (np.std(z) + 1e-12)
    return z * env

def rpm_profile(T, fs, rpm_lo, rpm_hi, drift_pct_rng, drift_period_rng, walk_std, transient_prob, rng):
    """
    Models time-varying propeller rotation speed with:
    - Slow sinusoidal drift
    - Random walk fluctuations
    - Occasional transient speed changes
    """
    n = int(T*fs)
    t = np.arange(n)/fs
    
    # Base RPM with sinusoidal drift
    rpm0 = rng.uniform(rpm_lo, rpm_hi)
    pct = rng.uniform(*drift_pct_rng)/100.0
    period = rng.uniform(*drift_period_rng)
    rpm_t = rpm0 * (1 + pct*np.sin(2*np.pi*t/period))
    
    # Add random walk (models load variations, control imperfections)
    walk = np.cumsum(rng.normal(0, walk_std, n))
    walk = walk - np.mean(walk)  # Zero mean
    rpm_t += walk
    
    # Add transient events (speed changes)
    n_transients = rng.binomial(int(T/10), transient_prob)  # Check every ~10s
    for _ in range(n_transients):
        t_start = rng.integers(0, max(1, n - int(3*fs)))
        delta_rpm = rng.uniform(-0.15, 0.15) * rpm0  # ±15% change
        transition_samples = int(rng.uniform(2, 8) * fs)  # 2-8 second transition
        ramp = np.linspace(0, 1, transition_samples)
        t_end = min(t_start + transition_samples, n)
        rpm_t[t_start:t_end] += delta_rpm * ramp[:t_end-t_start]
        if t_end < n:
            rpm_t[t_end:] += delta_rpm
    
    # Ensure RPM stays in reasonable bounds
    rpm_t = np.clip(rpm_t, rpm_lo*0.8, rpm_hi*1.2)
    
    return rpm_t

def bpf_tones(blades, rpm_t, harmonics, fs, amp_rng, rng):
    """
    Generates tonal components from propeller blade rates.
    Uses 1/k^1.5 decay (more realistic than 1/k for marine propellers).
    """
    n = len(rpm_t)
    f_bpf = blades * rpm_t / 60.0
    phase = 2*np.pi*np.cumsum(f_bpf)/fs
    y = np.zeros(n, dtype=float)
    base = rng.uniform(*amp_rng)
    for k in range(1, harmonics+1):
        # 1/k^1.5 decay is more realistic for marine propellers
        y += (base/(k**1.5)) * np.sin(k*phase + rng.uniform(0, 2*np.pi))
    return y

def rpm_dependent_cavitation_rate(rpm_t, rpm_threshold, base_rate, exponent):
    """
    Cavitation rate increases dramatically above threshold RPM.
    Physics: Cavitation number σ ~ 1/V^2, where V ~ RPM
    """
    rpm_ratio = np.maximum(rpm_t / rpm_threshold, 1.0)
    rate_multiplier = rpm_ratio ** exponent
    return base_rate * rate_multiplier

def speed_dependent_mixing(rpm_mean, rpm_lo, rpm_hi, class_name):
    """
    Adjust component mixing weights based on operating speed.
    At low speeds: More broadband flow noise
    At high speeds: More tonal and cavitation components
    """
    rpm_normalized = (rpm_mean - rpm_lo) / max(rpm_hi - rpm_lo, 1.0)
    rpm_normalized = np.clip(rpm_normalized, 0.0, 1.0)
    
    if class_name == "submarine":
        # Submarines: Stealth design, tonals suppressed at low speed
        w_broad = 0.75 - 0.15 * rpm_normalized  # 0.75 -> 0.60
        w_tip = 0.10 + 0.20 * (rpm_normalized**2)  # 0.10 -> 0.30 (quadratic, cavitation onset)
        w_hub = 0.10
        w_tonal = 0.05 + 0.10 * rpm_normalized  # 0.05 -> 0.15
    else:  # torpedo
        # Torpedoes: Higher speed, tonals more prominent
        w_broad = 0.50 - 0.10 * rpm_normalized  # 0.50 -> 0.40
        w_tip = 0.20 + 0.25 * rpm_normalized  # 0.20 -> 0.45
        w_hub = 0.15
        w_tonal = 0.15 + 0.20 * rpm_normalized  # 0.15 -> 0.35
    
    # Normalize to ensure sum ~ 1.0
    total = w_broad + w_tip + w_hub + w_tonal
    return w_broad/total, w_tip/total, w_hub/total, w_tonal/total

def apply_doppler_shift(s, fs, doppler_factor):
    """
    Apply Doppler shift to signal.
    doppler_factor = (c + v_r) / (c + v_s)
    where v_r = receiver velocity toward source (negative if away)
          v_s = source velocity away from receiver (negative if toward)
    For stationary receiver: doppler_factor ≈ 1 + v_s/c
    """
    if abs(doppler_factor - 1.0) < 1e-6:
        return s
    
    # Resample to simulate Doppler
    n_orig = len(s)
    n_new = int(n_orig / doppler_factor)
    t_orig = np.arange(n_orig) / fs
    t_new = np.arange(n_new) / (fs / doppler_factor)
    
    # Simple linear interpolation
    s_doppler = np.interp(t_new, t_orig, s)
    
    # Pad or truncate to original length for consistency
    if len(s_doppler) < n_orig:
        s_doppler = np.pad(s_doppler, (0, n_orig - len(s_doppler)), 'constant')
    else:
        s_doppler = s_doppler[:n_orig]
    
    return s_doppler

def synthesize(class_name, T, fs, params: ClassParams, rng, doppler_velocity_ms=None):
    """
    Combines all components into a final SOURCE signal with metadata.
    This signal is ready for Bellhop propagation (no channel effects applied here).
    
    doppler_velocity_ms: Source velocity in m/s (positive = away from receiver)
    """
    n = int(T*fs)
    blades = rng.integers(params.blades_lo, params.blades_hi+1)
    
    # Generate RPM profile with enhanced realism
    rpm_t = rpm_profile(T, fs, params.rpm_lo, params.rpm_hi,
                        params.rpm_drift_pct, params.rpm_drift_period_s,
                        params.rpm_walk_std, params.transient_prob, rng)
    
    rpm_mean = float(np.mean(rpm_t))
    
    # Generate tonal components
    tones = bpf_tones(blades, rpm_t, params.harmonics, fs, amp_rng=(0.005, 0.05), rng=rng)
    
    # Generate broadband and hub noise
    broad = colored_noise(rng.uniform(*params.alpha_broad), fs, params.broad, n, rng)
    hub = colored_noise(rng.uniform(*params.alpha_hub), fs, params.hub, n, rng)
    
    # RPM-dependent cavitation
    base_rate = rng.uniform(*params.tip_rate_base_hz)
    mean_rate = rpm_dependent_cavitation_rate(
        np.array([rpm_mean]), 
        params.tip_rpm_threshold, 
        base_rate, 
        params.tip_rate_exponent
    )[0]
    events = max(0, rng.poisson(mean_rate * T))
    tip = bursty_band_noise(fs, params.tip, n, events,
                           (params.tip_dur_ms[0]/1000.0, params.tip_dur_ms[1]/1000.0), rng)
    
    # Speed-dependent mixing
    w_broad, w_tip, w_hub, w_tonal = speed_dependent_mixing(
        rpm_mean, params.rpm_lo, params.rpm_hi, class_name
    )
    
    # Combine components with dynamic weights
    s = w_broad*broad + w_tip*tip + w_hub*hub + w_tonal*tones
    s /= (np.max(np.abs(s)) + 1e-9)
    
    # Apply Doppler shift if velocity specified
    doppler_factor = 1.0
    if doppler_velocity_ms is not None:
        c_water = 1500.0  # m/s
        doppler_factor = 1.0 + doppler_velocity_ms / c_water
        s = apply_doppler_shift(s, fs, doppler_factor)
    
    # Metadata
    meta = {
        "class": class_name,
        "blades": int(blades),
        "RPM_mean": float(rpm_mean),
        "RPM_std": float(np.std(rpm_t)),
        "BPF_mean_Hz": float(np.mean(blades * rpm_t / 60.0)),
        "cavitation_events": int(events),
        "cavitation_rate_hz": float(mean_rate),
        "mixing_broad": float(w_broad),
        "mixing_tonal": float(w_tonal),
        "mixing_tip": float(w_tip),
        "mixing_hub": float(w_hub),
        "doppler_factor": float(doppler_factor),
        "source_velocity_ms": float(doppler_velocity_ms) if doppler_velocity_ms else 0.0
    }
    
    return s.astype(np.float32), meta


## Pipline

In [4]:
from dataclasses import dataclass
import json, os
from pathlib import Path
import numpy as np
from scipy.io import wavfile
from scipy.signal import stft


def class_params_from_cfg(name, cfg):
    """Extract class parameters from config."""
    c = cfg["classes"][name]
    return ClassParams(
        blades_lo=int(c["blades"][0]), 
        blades_hi=int(c["blades"][1]),
        rpm_lo=float(c["rpm"][0]), 
        rpm_hi=float(c["rpm"][1]),
        broad=tuple(c["bands"]["broad_hz"]), 
        hub=tuple(c["bands"]["hub_hz"]), 
        tip=tuple(c["bands"]["tip_hz"]),
        alpha_broad=tuple(c["alpha_broad"]), 
        alpha_hub=tuple(c["alpha_hub"]),
        harmonics=int(c["bpf_harmonics"]),
        tip_rate_base_hz=tuple(c["tip_bursts"]["rate_base_hz"]),
        tip_rpm_threshold=float(c["tip_bursts"]["rpm_threshold"]),
        tip_rate_exponent=float(c["tip_bursts"]["rate_exponent"]),
        tip_dur_ms=tuple(c["tip_bursts"]["dur_ms"]),
        rpm_drift_pct=tuple(c["rpm_drift_pct"]), 
        rpm_drift_period_s=tuple(c["rpm_drift_period_s"]),
        rpm_walk_std=float(c["rpm_walk_std"]),
        transient_prob=float(c["transient_prob"]),
    )

def render_sample(class_name, cfg, out_dir, idx, rng, bellhop_mode=False):
    """
    Render a single sample.
    
    bellhop_mode: If True, outputs clean source signal only (no propagation/noise).
                  If False, applies simplified propagation and noise for Stage 1 validation.
    """
    # Sampling rate
    fs = cfg["fs"]["submarine"] if class_name=="submarine" else cfg["fs"]["torpedo"]
    
    # Duration
    dur_rng = cfg["duration_s"]["submarine"] if class_name=="submarine" else cfg["duration_s"]["torpedo"]
    T = rng.uniform(*dur_rng)
    
    # Doppler shift (if target is moving)
    doppler_vel = None
    if cfg.get("doppler", {}).get("enabled", False):
        vel_range = cfg["doppler"]["velocity_ms"]
        doppler_vel = rng.uniform(*vel_range)
    
    # Synthesize source signal
    params = class_params_from_cfg(class_name, cfg)
    s, meta_src = synthesize(class_name, T, fs, params, rng, doppler_velocity_ms=doppler_vel)
    
    # Environmental parameters (for metadata, used later in Bellhop)
    prop = cfg["propagation"]
    r = rng.uniform(*prop["ranges_m"])
    sd = rng.uniform(*prop["source_depth_m"])
    rd = rng.uniform(*prop["rx_depth_m"])
    ss = int(rng.integers(prop["sea_state"][0], prop["sea_state"][1]+1))
    water_depth = rng.uniform(*prop.get("water_depth_m", [100, 1000]))
    bottom_type = rng.choice(prop.get("bottom_types", ["sand", "mud", "rock"]))
    
    if bellhop_mode:
        # Output clean source signal only
        y = s
        actual_snr = None
    else:
        # Stage 1 validation: apply basic bandlimiting and noise
        # (In Stage 2, Bellhop will handle propagation properly)
        if class_name == "submarine":
            y = bandlimit(s, fs, 10, 1000)
        else:
            y = bandlimit(s, fs, 50, 10000)
        
        snr = float(rng.choice(cfg["sweeps"]["snr_dB"]))
        shipping = cfg.get("ambient", {}).get("shipping_level", 5)
        y = add_ambient_and_sensor_noise(y, fs, class_name, ss, snr, shipping, rng)
        actual_snr = snr
    
    # Save audio
    audio_dir = Path(out_dir)/"audio"
    audio_dir.mkdir(parents=True, exist_ok=True)
    fname = f"{idx:06d}_{class_name}.wav"
    wavfile.write(audio_dir/fname, fs, (np.clip(y, -1, 1) * 32767).astype(np.int16))
    
    # Generate spectrogram
    spec_dir = Path(out_dir)/"spec"
    spec_dir.mkdir(parents=True, exist_ok=True)
    f, t, S = stft(y, fs=fs, window="hann", 
                   nperseg=1024 if fs>=16000 else 512, 
                   noverlap=None, detrend=False, 
                   return_onesided=True, boundary=None, padded=False)
    
    try:
        import matplotlib.pyplot as plt
        import matplotlib
        matplotlib.use("Agg")
        SdB = 20*np.log10(np.abs(S)+1e-12)
        plt.figure(figsize=(8,4))
        plt.pcolormesh(t, f, SdB, shading="nearest", cmap="viridis")
        plt.colorbar(label="dB")
        plt.xlabel("Time (s)")
        plt.ylabel("Frequency (Hz)")
        plt.title(f"{class_name} - RPM: {meta_src['RPM_mean']:.1f}")
        plt.tight_layout()
        plt.savefig(spec_dir/f"{fname.replace('.wav','.png')}", dpi=120)
        plt.close()
    except Exception as e:
        print(f"Warning: Could not generate spectrogram: {e}")
    
    # Metadata
    md = {
        "class": class_name,
        "fs": int(fs),
        "duration_s": float(T),
        "range_m": float(r),
        "source_depth_m": float(sd),
        "rx_depth_m": float(rd),
        "water_depth_m": float(water_depth),
        "bottom_type": bottom_type,
        "sea_state": int(ss),
        "bellhop_mode": bellhop_mode,
    }
    
    if actual_snr is not None:
        md["snr_dB"] = float(actual_snr)
    
    # Add source metadata
    md.update(meta_src)
    
    return fname, md

def generate(cfg, out_dir, n, class_filter=None, seed=1337, bellhop_mode=False):
    """
    Generate dataset.
    
    Parameters:
    - cfg: Configuration dictionary
    - out_dir: Output directory
    - n: Number of samples to generate
    - class_filter: Generate only this class (None for alternating)
    - seed: Random seed
    - bellhop_mode: If True, output clean source signals for Bellhop Stage 2
    """
    rng = rng_from_seed(seed)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    
    metas = []
    for i in range(n):
        cls = class_filter if class_filter in ("submarine","torpedo") else ("submarine" if (i%2==0) else "torpedo")
        fname, md = render_sample(cls, cfg, out_dir, i, rng, bellhop_mode=bellhop_mode)
        metas.append({"file": str(Path("audio")/fname), **md})
    
    # Save metadata
    with open(Path(out_dir)/"metadata.jsonl","w") as f:
        for m in metas:
            f.write(json.dumps(m)+"\n")
    
    # Save config for reference
    import yaml
    with open(Path(out_dir)/"config_used.yaml", "w") as f:
        yaml.dump(cfg, f, default_flow_style=False)
    
    return metas

## Generate

In [5]:
import argparse, json, yaml
from pathlib import Path
import os


def main(config: str = 'enhanced.yaml',
         out: str = './testdata',
         n: int = 20,
         class_name: str=None,
         seed: int= 1337,
         bellhop_mode: str="store_true"):
    
    config_path = "./configs/" + config
    cfg = yaml.safe_load(open(config_path, 'r'))
    
    
    # Generate samples
    metas = generate(cfg,
                    out,
                    n, 
                    class_filter=class_name, 
                    seed=seed)


    
    # Summary
    print(f"\n{'='*60}")
    print(f"Generated {len(metas)} samples to {out}")
    print(f"Mode: {'Bellhop-ready (clean sources)' if bellhop_mode else 'Stage 1 validation (with noise)'}")
    print(f"{'='*60}\n")
    
    # Show first 3 samples
    print("First 3 metadata entries:")
    for i, m in enumerate(metas[:3], 1):
        print(f"\n--- Sample {i} ---")
        print(json.dumps(m, indent=2))
    
    # Class distribution
    classes = [m['class'] for m in metas]
    print(f"\n{'='*60}")
    print(f"Class distribution:")
    print(f"  Submarine: {classes.count('submarine')}")
    print(f"  Torpedo: {classes.count('torpedo')}")
    print(f"{'='*60}\n")

In [6]:
main()


Generated 20 samples to ./testdata
Mode: Bellhop-ready (clean sources)

First 3 metadata entries:

--- Sample 1 ---
{
  "file": "audio/000000_submarine.wav",
  "class": "submarine",
  "fs": 16000,
  "duration_s": 37.562038006942366,
  "range_m": 6533.877916648885,
  "source_depth_m": 78.30109265206211,
  "rx_depth_m": 75.13232265170018,
  "water_depth_m": 911.3274867732522,
  "bottom_type": "clay",
  "sea_state": 4,
  "bellhop_mode": false,
  "snr_dB": 5.0,
  "blades": 6,
  "RPM_mean": 127.00837434807273,
  "RPM_std": 82.99509165233955,
  "BPF_mean_Hz": 12.700837434807282,
  "cavitation_events": 2,
  "cavitation_rate_hz": 0.11384424062872259,
  "mixing_broad": 0.644056517057787,
  "mixing_tonal": 0.10231626825247915,
  "mixing_tip": 0.1569568002125817,
  "mixing_hub": 0.09667041447715222,
  "doppler_factor": 0.9991403728218345,
  "source_velocity_ms": -1.2894407672481312
}

--- Sample 2 ---
{
  "file": "audio/000001_torpedo.wav",
  "class": "torpedo",
  "fs": 24000,
  "duration_s": 10